In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
DATA_PATH = "../data/raw/telco_customer_churn.csv"
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)

Dataset shape: (7043, 33)


In [ ]:
TARGET = "Churn Label"
y = df[TARGET].map({"No": 0,"Yes": 1})

In [ ]:
X = df.drop(columns=[TARGET]).copy()

columns_to_drop = ["Count","Country","State","CustomerID","Churn Value","Churn Score","Churn Reason"]

In [ ]:
X = X.drop(columns=columns_to_drop)
X["Total Charges"] = pd.to_numeric(X["Total Charges"],errors="coerce")

In [ ]:
X.loc[(X["Tenure Months"] == 0) &(X["Total Charges"].isna()),"Total Charges"] = 0
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 25)
y shape: (7043,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

In [ ]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

X_train: (5634, 25)
X_test : (1409, 25)


In [ ]:
numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()

In [ ]:
categorical_features = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

In [ ]:
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

Numerical features: 7
Categorical features: 18


In [ ]:
numerical_pipeline = Pipeline(steps=[("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])

In [ ]:
categorical_pipeline = Pipeline(steps=[("imputer" ,SimpleImputer(strategy="most_frequent")) ,("encoder" ,OneHotEncoder(handle_unknown = "ignore", sparse_output=False))])

In [ ]:
preprocessor = ColumnTransformer(transformers=[("num",numerical_pipeline,numerical_features),("cat",categorical_pipeline,categorical_features)])

In [ ]:
logistic_pipeline = Pipeline(steps=[("preprocessor",preprocessor),("model",LogisticRegression(max_iter=1000,random_state=42))])

In [ ]:
print("Training Logistic Regression...")
logistic_pipeline.fit(X_train,y_train)
print("Logistic Regression training completed.")

Training Logistic Regression...
Logistic Regression training completed.


In [ ]:
decision_tree_preprocessor = ColumnTransformer(transformers=[("num",numerical_pipeline,numerical_features),("cat",categorical_pipeline,categorical_features)])

In [ ]:
decision_tree_pipeline = Pipeline(steps=[("preprocessor",decision_tree_preprocessor),("model",DecisionTreeClassifier(random_state=42))])

In [ ]:
decision_tree_pipeline.fit(X_train,y_train)
print("Decision Tree training completed.")

Decision Tree training completed.


In [ ]:
random_forest_preprocessor = ColumnTransformer(transformers=[("num",numerical_pipeline,numerical_features),("cat",categorical_pipeline,categorical_features)])

In [ ]:
random_forest_pipeline = Pipeline(steps=[("preprocessor",random_forest_preprocessor),("model",RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1))])

In [ ]:
random_forest_pipeline.fit(X_train,y_train)
print("Random Forest training completed.")

Random Forest training completed.


In [ ]:
logistic_predictions = logistic_pipeline.predict(X_test)
decision_tree_predictions = decision_tree_pipeline.predict(X_test)
random_forest_predictions = random_forest_pipeline.predict(X_test)
print("Predictions generated successfully.")

Predictions generated successfully.


In [ ]:
print("Logistic Regression:")
print(pd.Series(logistic_predictions).value_counts())
print("\nDecision Tree:")
print(pd.Series(decision_tree_predictions).value_counts())
print("\nRandom Forest:")
print(pd.Series(random_forest_predictions).value_counts())

Logistic Regression:
0    1085
1     324
Name: count, dtype: int64

Decision Tree:
0    1064
1     345
Name: count, dtype: int64

Random Forest:
0    1138
1     271
Name: count, dtype: int64
